In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))

import pandas as pd
import inventory as I
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
base = Path.home() / "Documents/DS217/users/jormond"
open_picu_dir = base / "open_picu"
gosh_dir = base / "GOSH"

I.list_files(gosh_dir)

In [ ]:
gosh = I.inventory_site(gosh_dir, recursive=True, exclude=r"lab_components", n_jobs=3)
I.save_inventory(gosh, open_picu_dir/"derived"/"gosh_inventory.pkl")
gosh["summary"].sort_values("seconds", ascending=False).head(10)

In [ ]:
raw = I.inventory_site(gosh_dir/"raw_lab_values", recursive=False,
                       exclude=r"^lab_variables_(?!INR|Glucose).*lab_components")

In [ ]:
base = "PICANet_All_patients_SR_011024_projectID.csv"
pic = ["Archived/picanet_file_with_mapping.csv", base,
       "PICANet_ALL_patients_SR_011024_projectID_eprweights.csv",
       "gosh_mech_vent_31March2026/final_gosh_dataset.parquet"]

d = gosh["dates"].copy()
d["earliest"] = pd.to_datetime(d["earliest"])
d["latest"] = pd.to_datetime(d["latest"])
dsum = d.groupby("column").agg(files=("file", "count"), earliest=("earliest", "min"), latest=("latest", "max"))

parts = ["# 1. Schema groups", I.df_text(I.schema_groups(gosh), index=False)]
for fn in pic:
    parts += [f"# 2. Profile: {fn}", I.df_text(gosh["profiles"][fn], index=False)]
parts += [f"# 3. ID overlap with {base}", I.df_text(I.id_overlap(gosh, base=base), index=False)]
parts += ["# 4. Dates by column", I.df_text(dsum)]
parts += ["# 5. Logs and notes (each cut at 4,000 characters)"]
for p in sorted(gosh_dir.rglob("*.txt")):
    text = p.read_text(errors="replace")
    parts += [f"## {p.relative_to(gosh_dir).as_posix()} ({len(text)} characters)", text[:4000]]

out = open_picu_dir/"derived"/"gosh_for_claude.md"
out.write_text("\n\n".join(parts))
print(out, round(out.stat().st_size / 1e3), "KB")

In [ ]:
import importlib, gosh_links as GL
importlib.reload(GL)

GL.candidates(gosh_dir, [r"picanet", r"heart|pulse|^HR|/HR", r"\.parquet$"])

In [ ]:
gosh_dir


In [ ]:
gosh_links = GL.check_links(
    gosh_dir,
    picanet=gosh_dir/"PICANet_All_patients_SR_011024_projectID.csv",
    picanet_epr=gosh_dir/"PICANet_ALL_patients_SR_011024_projectID_eprweights.csv",
    archived=gosh_dir/"Archived/picanet_file_with_mapping.csv",
    hr_file=gosh_dir/ "merged_values" / "HR_merged.csv",
    parquet=gosh_dir/"gosh_mech_vent_31March2026/final_gosh_dataset.parquet",
)

In [ ]:
importlib.reload(GL)
GL.check_windows(
    gosh_dir,
    archived="Archived/picanet_file_with_mapping.csv",
    parquet="gosh_mech_vent_31March2026/final_gosh_dataset.parquet",
    links=gosh_links,
)

In [ ]:
importlib.reload(GL)
GL.candidates(gosh_dir, [r"merged_values/.*merged\.csv$"])

In [ ]:
vit = GL.vitals_coverage(
    gosh_dir,
    archived="Archived/picanet_file_with_mapping.csv",
    files=[
        "merged_values/HR_merged.csv",
        "merged_values/PulseRate_merged.csv",
        "merged_values/RespiratoryRate_merged.csv",
        "merged_values/SpO2_merged.csv",
        "merged_values/Temperature_merged.csv",
        "merged_values/Invasive_systolic_BP_merged.csv",
        "merged_values/Invasive_diastolic_BP_merged.csv",
        "merged_values/Invasive_mean_BP_merged.csv",
        "merged_values/Non-Invasive_systolic_BP_merged.csv",
        "merged_values/Non-Invasive_diastolic_BP_merged.csv",
        "merged_values/Non-Invasive_mean_BP_merged.csv",
    ],
)

In [ ]:
importlib.reload(GL)
cmp = GL.compare_picanet(
    gosh_dir,
    picanet="PICANet_All_patients_SR_011024_projectID.csv",
    picanet_epr="PICANet_ALL_patients_SR_011024_projectID_eprweights.csv",
    archived="Archived/picanet_file_with_mapping.csv",
)

In [ ]:
importlib.reload(GL)
oc = GL.compare_outcomes(
    gosh_dir,
    picanet="PICANet_All_patients_SR_011024_projectID.csv",
    archived="Archived/picanet_file_with_mapping.csv",
)

In [ ]:
importlib.reload(GL)
st = GL.compare_status(
    gosh_dir,
    picanet="PICANet_All_patients_SR_011024_projectID.csv",
    archived="Archived/picanet_file_with_mapping.csv",
)

In [ ]:
importlib.reload(GL)
vit_main = GL.vitals_coverage(
    gosh_dir,
    archived="PICANet_ALL_patients_SR_011024_projectID_eprweights.csv",
    start_col="AdDateTime",
    end_col="UnitDisDateTime",
    files=[
        "merged_values/HR_merged.csv",
        "merged_values/PulseRate_merged.csv",
        "merged_values/RespiratoryRate_merged.csv",
        "merged_values/SpO2_merged.csv",
        "merged_values/Temperature_merged.csv",
        "merged_values/Invasive_systolic_BP_merged.csv",
        "merged_values/Invasive_diastolic_BP_merged.csv",
        "merged_values/Invasive_mean_BP_merged.csv",
        "merged_values/Non-Invasive_systolic_BP_merged.csv",
        "merged_values/Non-Invasive_diastolic_BP_merged.csv",
        "merged_values/Non-Invasive_mean_BP_merged.csv",
    ],
)